##### Build Session Results Fact Table (`facts_session_results`)

This notebook builds the **Session Results Fact Table** in the gold layer of our Formula 1 data lakehouse. It is part of the **medallion architecture** pipeline (bronze -> silver -> gold) and sits in the `04-gold` folder.

##### What this notebook does:
1. **Reads** the `results` and `sprints` tables from the silver layer
2. **Adds** a new column `session_type` to distinguish between "Race" and "Sprint" sessions
3. **Unions** both DataFrames into a single combined DataFrame
4. **Derives** additional analytical columns: `is_win`, `is_podium`, `has_points`
5. **Writes** the result as a Delta table to `formula1.gold.facts_session_results`

##### Purpose:
The `facts_session_results` table is a **unified fact table** that combines race results and sprint results into a single table. It adds derived boolean columns that make common analytical queries simpler (e.g., counting wins, podiums, or points finishes without needing CASE expressions at query time).

##### Dependencies:
- **Upstream config:** **`00-common/01.environment-config`** (provides `catalog_name`, `silver_schema`, `gold_schema` variables)
- **Source tables:** `formula1.silver.results`, `formula1.silver.sprints`
- **Target table:** `formula1.gold.facts_session_results`

##### Step 0: Load Environment Config & Import Functions
We run the shared environment configuration notebook which defines key variables:
- `catalog_name` -> `formula1`
- `silver_schema` -> `silver`
- `gold_schema` -> `gold`

Then we import all built-in functions from `pyspark.sql.functions` for transformations like `lit()`, `when()`, and `col()`.

In [0]:
%run ../00-common/01.environment-config 

##### Define the Gold Target Table
We define the fully qualified target table name where our fact table will be written. The variable `target_table` is set to `formula1.gold.facts_session_results` by combining:
- `catalog_name` -> the Unity Catalog name (`formula1`)
- `gold_schema` -> the gold layer schema (`gold`)
- Table name -> `facts_session_results`

This follows the **medallion architecture** (bronze -> silver -> gold), where the gold layer holds clean, business-ready fact and dimension tables.

In [0]:
from pyspark.sql.functions import *

##### Define the Gold Target Table
We define the fully qualified target table name where our fact table will be written. The variable `target_table` is set to `formula1.gold.facts_session_results` by combining:
- `catalog_name` -> `formula1`
- `gold_schema` -> `gold`
- Table name -> `facts_session_results`

In [0]:
target_table = f'{catalog_name}.{gold_schema}.facts_session_results'

##### Step 1: Read Source Tables & Add Session Type
We read both source tables from the silver layer and immediately enrich them with a `session_type` column to identify which type of session each row represents:

- `results_df` -> reads `formula1.silver.results`, adds `session_type = 'RACE'`, and drops unwanted metadata columns (`race_name`, `race_date`, `ingestion_timestamp`, `source_file`)
- `sprints_df` -> reads `formula1.silver.sprints`, adds `session_type = 'SPRINT'`, and drops the same metadata columns

The `lit()` function creates a constant/literal value column. After this step, both DataFrames have the same schema with an additional `session_type` column, making them ready for a union.

**Remaining columns after drop:** `round`, `season`, `constructor_id`, `driver_id`, `grid_position`, `Completed_laps`, `car_number`, `points`, `final_position`, `final_position_text`, `status`, `session_type`

In [0]:
results_df = (spark.table(f'{catalog_name}.{silver_schema}.results')
              .withColumn('session_type', lit('RACE'))
              .drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file')
)

In [0]:
sprints_df = (spark.table(f'{catalog_name}.{silver_schema}.sprints')
              .withColumn('session_type', lit('SPRINT'))
              .drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file')
)

##### Step 2: Combine Both Tables (Union)
We use `.unionByName()` to stack `results_df` and `sprints_df` vertically into a single DataFrame called `results_sprints_df`. This method matches columns **by name** (not position), ensuring data aligns correctly even if column order differs between the two tables.

After the union:
- Rows from `silver.results` have `session_type = 'RACE'`
- Rows from `silver.sprints` have `session_type = 'SPRINT'`
- All rows share the same schema

We then `display()` the combined DataFrame to verify the union worked correctly.

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)
display(results_sprints_df)

##### Step 3: Derive Additional Analytical Columns
We create a new DataFrame `facts_session_results_df` by adding three derived boolean columns to `results_sprints_df`:

| Column | Logic | Description |
|--------|-------|-------------|
| `is_win` | `final_position == 1` | `true` if the driver won this session |
| `is_podium` | `final_position BETWEEN 1 AND 3` | `true` if the driver finished in the top 3 |
| `has_points` | `points > 0` | `true` if the driver scored points in this session |

These derived columns eliminate the need for CASE/WHEN expressions in dashboards and queries - analysts can simply `SUM(is_win)` or filter `WHERE is_podium = true`.

We also verify the output by filtering for `season == 2025` to spot-check recent results.

In [0]:
facts_session_results_df =(results_sprints_df
 .withColumn('is_win',col('final_position') == 1)
 .withColumn('is_podium', col('final_position').between(1,3))
 .withColumn('has_points', col('points') > 0))


In [0]:
display(facts_session_results_df.filter('season == 2025'))

##### Step 4: Write to Gold Layer
Finally, we write the `facts_session_results_df` DataFrame to the gold schema as a Delta table. This is the **central fact table** for all race and sprint performance analysis.

- **Format:** Delta (supports ACID transactions, time travel, and schema evolution)
- **Mode:** Overwrite (replaces the entire table with fresh data each run)
- **Target:** `formula1.gold.facts_session_results`
- **Method:** `.saveAsTable(target_table)` creates a managed Unity Catalog table

Once written, this table can be joined with dimension tables (`dim_drivers`, `dim_constructors`, `dim_races`) for full analytical queries.

In [0]:
(
    facts_session_results_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(target_table)
)